In [7]:
import os
import json
from tqdm import tqdm
import torch
from torchvision import transforms
from torchvision.models.detection import maskrcnn_resnet50_fpn
from PIL import Image
import numpy as np
from skimage import measure

# Paths
MODEL_PATH = "outputs/maskrcnn_resnet50_fpn_20250609_1525/model_best.pth"
IMAGE_ROOT = "/mnt/c/Users/Jorge/Desktop/UNI/Tese/datasets/labelstudio_storage/flat_dataset"
OUTPUT_JSON = "maskrcnn_bubble_annotations.json"

MAX_IMAGES = 2000
CONFIDENCE_THRESHOLD = 0.95

In [8]:
import shutil

SRC_DIR = "/mnt/c/Users/Jorge/Desktop/UNI/Tese/datasets/chosen_images"
DST_DIR = "/mnt/c/Users/Jorge/Desktop/UNI/Tese/datasets/labelstudio_storage/flat_dataset"

# Clear destination folder
for filename in os.listdir(DST_DIR):
    file_path = os.path.join(DST_DIR, filename)
    if os.path.isfile(file_path):
        os.remove(file_path)

print(f"Cleared files in {DST_DIR}")

# Find images in source folder
image_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
image_paths = []
for dirpath, _, filenames in os.walk(SRC_DIR):
    for fname in filenames:
        if fname.lower().endswith(image_exts):
            image_paths.append(os.path.join(dirpath, fname))

# Copy first MAX_IMAGES files
for img_path in tqdm(image_paths[:MAX_IMAGES], desc="Copying files"):
    dst_path = os.path.join(DST_DIR, os.path.basename(img_path))
    shutil.copy2(img_path, dst_path)

print(f"Copied {min(MAX_IMAGES, len(image_paths))} images to {DST_DIR}")


Cleared files in /mnt/c/Users/Jorge/Desktop/UNI/Tese/datasets/labelstudio_storage/flat_dataset


Copying files:   0%|          | 0/250 [00:00<?, ?it/s]

Copying files: 100%|██████████| 250/250 [00:05<00:00, 42.33it/s]

Copied 250 images to /mnt/c/Users/Jorge/Desktop/UNI/Tese/datasets/labelstudio_storage/flat_dataset


In [9]:
# Prepare device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
model = maskrcnn_resnet50_fpn(pretrained=False, num_classes=2)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

# Image preprocessing
transform = transforms.Compose([
    transforms.ToTensor()
])

# Helper: find all images
def find_images(root):
    image_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
    image_paths = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith(image_exts):
                image_paths.append(os.path.join(dirpath, fname))
    return image_paths

# Helper: convert binary mask to polygons
def binary_mask_to_polygon(binary_mask):
    contours = measure.find_contours(binary_mask, 0.5)
    segmentation = []
    for contour in contours:
        contour = np.flip(contour, axis=1)  # (y,x) -> (x,y)
        segmentation.append(contour)
    return segmentation

# Start processing
images = find_images(IMAGE_ROOT)

tasks = []

for img_id, img_path in enumerate(tqdm(images)):

    img = Image.open(img_path).convert("RGB")
    width, height = img.size
    img_tensor = transform(img).to(device)

    with torch.no_grad():
        prediction = model([img_tensor])[0]

    boxes = prediction["boxes"]
    masks = prediction["masks"]  # (N, 1, H, W)
    scores = prediction["scores"]
    labels = prediction["labels"]

    # Compute mask areas (sum over H and W)
    areas = masks.sum(dim=[1, 2, 3])  # Sum over (1, H, W)

    # Apply confidence and area filters
    MIN_AREA = 1000  # adjust to what you consider small (in pixels)

    keep = (scores > CONFIDENCE_THRESHOLD) & (areas > MIN_AREA)

    boxes = boxes[keep].cpu()
    masks = (masks[keep].cpu() > 0.5).squeeze(1)  # Binarize and remove channel dim
    scores = scores[keep].cpu()
    labels = labels[keep].cpu()


    results = []

    for box, mask, label, score in zip(boxes, masks, labels, scores):
        mask_bin = mask.numpy()
        polygons = binary_mask_to_polygon(mask_bin)

        for poly in polygons:
            if len(poly) < 3:
                continue  # skip degenerate

            points = [[(x / width * 100), (y / height * 100)] for x, y in poly]

            results.append({
                "original_width": width,
                "original_height": height,
                "image_rotation": 0,
                "value": {
                    "points": points,
                    "polygonlabels": ["bubble"]
                },
                "from_name": "label",
                "to_name": "image",
                "type": "polygon"
            })


    if results:
        tasks.append({
            "data": {
                "image": f"/data/local-files/?d={img_path}"
            },
            "annotations": [
                {
                    "result": results
                }
            ]
        })

# Save tasks
with open(OUTPUT_JSON, "w") as f:
    json.dump(tasks, f, indent=2)

print(f"Label Studio task JSON saved to: {OUTPUT_JSON}")


100%|██████████| 250/250 [01:33<00:00,  2.66it/s]


Label Studio task JSON saved to: maskrcnn_bubble_annotations.json
